# Model Evaluation Metrics

**When:** Week 13 · Session A  
**Goal:** Judge models correctly — not just "accuracy looks high."

### Classification metrics
- Accuracy, Precision, Recall, F1
- Confusion matrix
- ROC-AUC (intro)

### Regression metrics
- MAE, MSE, RMSE, R²

### Cross-validation
- Why one train/test split is not enough


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_absolute_error, mean_squared_error, r2_score, RocCurveDisplay
)


## 1. Classification demo (breast cancer)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000)),
])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1       :", round(f1_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))


**Intuition**
- **Precision:** of predicted positives, how many were correct?
- **Recall:** of real positives, how many did we catch?
- Medical screening often cares more about **recall** (don't miss disease).
- Spam filters often care more about **precision** (don't block real email).


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=data.target_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

RocCurveDisplay.from_estimator(clf, X_test, y_test)
plt.title("ROC Curve")
plt.show()


## 2. Regression metrics (California housing)

In [ ]:
housing = fetch_california_housing()
Xh, yh = housing.data, housing.target
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=42)

reg = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
reg.fit(Xh_train, yh_train)
yh_pred = reg.predict(Xh_test)

mae = mean_absolute_error(yh_test, yh_pred)
rmse = mean_squared_error(yh_test, yh_pred) ** 0.5
r2 = r2_score(yh_test, yh_pred)
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")


## 3. Cross-validation

In [ ]:
scores = cross_val_score(clf, X, y, cv=5, scoring="f1")
print("F1 per fold:", np.round(scores, 3))
print("Mean F1    :", round(scores.mean(), 3), "+/-", round(scores.std(), 3))


## Practice
1. Reload `heart.csv` or `diabetes.csv` from class.
2. Train logistic regression.
3. Report accuracy, precision, recall, F1 and plot the confusion matrix.
4. Run 5-fold cross-validation and compare to the single split score.


In [ ]:
from pathlib import Path
heart_path = Path("../data_cleaningML/data_cleaning/heart.csv")
if heart_path.exists():
    heart = pd.read_csv(heart_path)
    print(heart.head())
    # Students: identify target column, then train + evaluate
else:
    print("heart.csv not found — use breast cancer example above for practice.")
